# Bispectrum-window grid convergence

This notebook only reads products written by `full_shape/job_scripts/study_bispectrum_window.py`. Expensive likelihood, direct-theory, native-window, Fisher, and profile evaluations are performed by the script.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

RESULT_ROOT = Path('/pscratch/sd/e/epaillas/bk_window_convergence/lrg2')
plt.style.use('tableau-colorblind10')

In [ ]:
def read_optional(path):
    return pd.read_csv(path) if path.exists() and path.stat().st_size else pd.DataFrame()

products = {}
for directory in sorted(RESULT_ROOT.glob('observable_dk_*')):
    manifest = json.loads((directory / 'manifest.json').read_text())
    observable_dk = manifest['observable_dk']
    products[observable_dk] = {
        'manifest': manifest,
        **{name: read_optional(directory / f'{name}.csv') for name in [
            'evaluations', 'metrics', 'convergence', 'reweighting',
            'emulator_validation', 'native_validation', 'fisher_bias', 'profiles']},
    }
print('Loaded observable grids:', sorted(products))

## Prediction and likelihood convergence

`model_snr` is the covariance-weighted norm $\sqrt{\Delta m^T\Psi\Delta m}$. `delta_chi2` is the actual same-parameter likelihood change and can have either sign.

In [ ]:
summary_rows = []
for observable_dk, product in products.items():
    frame = product['metrics']
    if frame.empty:
        continue
    frame = frame.query("block == 'bispectrum'")
    for grid, group in frame.groupby('grid'):
        summary_rows.append({
            'observable_dk': observable_dk,
            'theory_grid': grid,
            'npoints': len(group),
            'median_model_snr': group.model_snr.median(),
            'p95_model_snr': group.model_snr.quantile(0.95),
            'max_model_snr': group.model_snr.max(),
            'median_abs_delta_chi2': group.delta_chi2.abs().median(),
            'p95_abs_delta_chi2': group.delta_chi2.abs().quantile(0.95),
        })
summary = pd.DataFrame(summary_rows)
display(summary)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for observable_dk, product in products.items():
    frame = product['metrics'].query("block == 'bispectrum'")
    for grid, group in frame.groupby('grid'):
        label = rf'$dk_{{obs}}={observable_dk:g}$, $dk_{{th}}={grid}$'
        axes[0].hist(group.model_snr, bins=30, histtype='step', density=True, label=label)
        axes[1].hist(group.delta_chi2, bins=30, histtype='step', density=True, label=label)
axes[0].set(xlabel=r'$\sqrt{\Delta m^T\Psi\Delta m}$', ylabel='density')
axes[1].set(xlabel=r'$\Delta\chi^2(\theta)$', ylabel='density')
for ax in axes:
    ax.grid(alpha=0.25)
axes[1].legend(fontsize=8)
fig.tight_layout()

## Successive-grid behavior and inference proxies

In [ ]:
for observable_dk, product in products.items():
    print(f'Observable dk = {observable_dk:g}')
    convergence = product['convergence']
    if not convergence.empty:
        display(convergence.query("block == 'bispectrum'")[
            ['convergence_ratio', 'convergence_order', 'monotonic']].describe())
    for name in ['native_validation', 'emulator_validation', 'reweighting', 'fisher_bias', 'profiles']:
        frame = product[name]
        if not frame.empty:
            print(name)
            display(frame.head(20))

## Interpretation checklist

- First verify that the compact $dk_{th}=0.0025$ result agrees with the fully native anchors.
- Verify emulator errors are smaller than window-grid differences.
- Look for a reduction in both `model_snr` and $|\Delta\chi^2|$ from 0.01→0.005→0.0025, and inspect points flagged as non-monotonic.
- Treat importance-reweighted shifts as reliable only when `weight_ess_fraction` remains healthy and opposite-direction reweighting agrees.
- Use profile and Fisher shifts as complementary checks; the profile comparison is authoritative when the local approximation fails.
- Do not compare raw $\chi^2$ values across different observable binning dimensions; compare theory-grid choices within each `observable_dk` directory.